In [8]:
import pandas as pd
import io

def custom_parse(csv_string):
    """Custom parser to handle the '100,358' issue."""
    lines = csv_string.splitlines()
    rows = []
    for line in lines:
        row = line.split(',')
        if len(row) == 7:  # Correct number of columns
            rows.append(row)
        elif len(row) > 7:
            # Combine the last two elements if they were split
            rows.append(row[:6] + [','.join(row[6:])])
    return io.StringIO('\n'.join([','.join(r) for r in rows]))

# Load the target CSV with custom parser
with open("rocksdb_benchmark_results_sequential.csv", 'r') as f:
    parsed_csv = custom_parse(f.read())

target_df = pd.read_csv(parsed_csv, index_col=False)

# Add the new column
target_df['number_of_operations'] = pd.NA

# Define the pattern of number_of_operations values
number_of_operations_values = [100, 1000, 10000, 100000]

# Iterate through the DataFrame and apply the pattern
current_index = 0
pattern_index = 0

while current_index < len(target_df):
    number_of_operations = number_of_operations_values[pattern_index % len(number_of_operations_values)]
    end_index = min(current_index + number_of_operations, len(target_df))

    target_df.iloc[current_index:end_index, target_df.columns.get_loc('number_of_operations')] = number_of_operations

    current_index = end_index
    pattern_index += 1

# Save the backfilled CSV
target_df.to_csv("rocksdb_benchmark_results_sequential_backfilled.csv", index=False)
print("Backfilled CSV saved to rocksdb_benchmark_results_sequential_backfilled.csv")

# Verify data integrity by logging boundary values
current_index = 0
pattern_index = 0

while current_index < len(target_df):
    number_of_operations = number_of_operations_values[pattern_index % len(number_of_operations_values)]
    end_index = min(current_index + number_of_operations, len(target_df))

    # Log boundary values
    if current_index > 0:
        if current_index - 1 < len(target_df):
            print(f"Row {current_index - 1}: {target_df.iloc[current_index - 1]}")
    if current_index < len(target_df):
        print(f"Row {current_index}: {target_df.iloc[current_index]}")
    if end_index < len(target_df):
        print(f"Row {end_index}: {target_df.iloc[end_index]}")

    current_index = end_index
    pattern_index += 1

/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_5837/482376534.py:21: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  target_df = pd.read_csv(parsed_csv, index_col=False)


Backfilled CSV saved to rocksdb_benchmark_results_sequential_backfilled.csv
Row 0: data_size               104857600
operation_type                GET
write_buffer_size              2M
block_cache_size             128M
compaction_style            level
bloom_filter_policy          True
latency                       100
number_of_operations          100
Name: 0, dtype: object
Row 100: data_size               104857600
operation_type                GET
write_buffer_size              2M
block_cache_size             128M
compaction_style            level
bloom_filter_policy          True
latency                      1000
number_of_operations         1000
Name: 100, dtype: object
Row 99: data_size               104857600
operation_type                GET
write_buffer_size              2M
block_cache_size             128M
compaction_style            level
bloom_filter_policy          True
latency                       100
number_of_operations          100
Name: 99, dtype: object
Row 100: dat